In [ ]:
import yfinance as yf
import pandas as pd
from pathlib import Path

# BIST30 tickers (yfinance requires .IS suffix)
BIST30_TICKERS = [
    "AKBNK.IS", "ARCLK.IS", "ASELS.IS", "BIMAS.IS", "CIMSA.IS",
    "EKGYO.IS", "EREGL.IS", "FROTO.IS", "GARAN.IS", "GUBRF.IS",
    "HALKB.IS", "ISCTR.IS", "KCHOL.IS", "KONTR.IS", "KRDMD.IS",
    "MGROS.IS", "ODAS.IS", "PETKM.IS", "PGSUS.IS", "SAHOL.IS",
    "SASA.IS", "SISE.IS", "TAVHL.IS", "TCELL.IS", "THYAO.IS",
    "TKFEN.IS", "TOASO.IS", "TUPRS.IS", "VAKBN.IS", "YKBNK.IS"
]

print(f"Total tickers: {len(BIST30_TICKERS)}")
print(BIST30_TICKERS[:5])

In [ ]:
!pip install pyarrow

In [ ]:
raw_dir = Path("../data/raw/prices")
raw_dir.mkdir(parents=True, exist_ok=True)

all_data = {}
failed = []

for ticker in BIST30_TICKERS:
    try:
        df = yf.download(ticker, period="5y", auto_adjust=True, progress=False)
        if len(df) < 100:
            failed.append(ticker)
            continue
        df.to_parquet(raw_dir / f"{ticker.replace('.IS', '')}.parquet")
        all_data[ticker] = df
        print(f"✅ {ticker}: {len(df)} days")
    except Exception as e:
        failed.append(ticker)
        print(f"❌ {ticker}: {e}")

print(f"\nSuccess: {len(all_data)}, Failed: {failed}")

In [ ]:
# Quick check - THYAO example
thyao = pd.read_parquet("../data/raw/prices/THYAO.parquet")

print("Shape:", thyao.shape)
print("Date range:", thyao.index[0].date(), "→", thyao.index[-1].date())
print("Missing values:\n", thyao.isnull().sum())
thyao.head()

In [ ]:
summary = []
for ticker in BIST30_TICKERS:
    name = ticker.replace('.IS', '')
    try:
        df = pd.read_parquet(f"../data/raw/prices/{name}.parquet")
        summary.append({
            "ticker": name,
            "n_days": len(df),
            "start": df.index[0].date(),
            "end": df.index[-1].date(),
            "missing": df.isnull().sum().sum()
        })
    except:
        pass

summary_df = pd.DataFrame(summary).sort_values("n_days")
print(f"Total stocks: {len(summary_df)}")
print(f"Any missing values: {summary_df['missing'].sum()}")
summary_df

let's making eda !!

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Load all closing prices into one dataframe
close_prices = {}
for ticker in BIST30_TICKERS:
    name = ticker.replace('.IS', '')
    df = pd.read_parquet(f"../data/raw/prices/{name}.parquet")
    close_prices[name] = df["Close"].iloc[:, 0]  # ilk kolonu al

close_df = pd.DataFrame(close_prices)
print("Shape:", close_df.shape)
close_df.head()

In [ ]:
# Normalize prices to 100 (to compare performance across stocks)
normalized = close_df.div(close_df.iloc[0]) * 100

plt.figure(figsize=(14, 6))
for col in normalized.columns:
    plt.plot(normalized.index, normalized[col], alpha=0.4, linewidth=0.8)

plt.title("BIST30 - Normalized Price Performance (Base=100)", fontsize=14)
plt.xlabel("Date")
plt.ylabel("Normalized Price")
plt.axhline(y=100, color='white', linestyle='--', linewidth=0.8)
plt.tight_layout()
plt.show()

## BIST30 Normalized Price Performance

All stocks normalized to base 100 (starting price = 100) to enable fair comparison 
across different price levels.

**Key observations:**
- 2021–2022: Relatively flat movement across all stocks
- 2022 onwards: Sharp nominal price increases driven by Turkey's high inflation environment
- Top performer reached ~5800x base — extreme outlier likely driven by sector-specific catalysts
- Most stocks clustered between 500–2000x, reflecting broad market inflation

In [ ]:
# Daily returns
# This will be the basis of anomaly detection 
returns = close_df.pct_change().dropna()

print("Returns shape:", returns.shape)
print("\nDescriptive stats:")
returns.describe().round(4)